In [0]:
%python
pip install openpyxl

In [0]:
%python
import re
import pandas as pd
from functools import reduce
from pyspark.sql import functions as F

raw = "/Volumes/clinical_portfolio/bronze/raw_files"

SOURCES = [
    ("metadata", ["MetaData"], "metadata"),
    ("bal", ["Baseline", "Month3", "Month6", "Month12"], "bal"),
    ("pbmc", ["Baseline", "Month3", "Month6", "Month12"], "pbmc"),
    ("luminex", ["Unstimulated", "ESAT6_CFP10", "PPD", "PHA"], "luminex_raw"),
    ("luminex", ["Selection_Criteria"], "luminex_selection"),
    ("treatment", ["Treatment"], "treatment"),
]

def snake(col: str) -> str:
    col = col.strip().lower().replace("%", "_pct")
    return re.sub(r"[^0-9a-z]+", "_", col).strip("_")

def load_sheet(stem: str, sheet: str):
    pdf = pd.read_excel(f"{raw}/{stem}.xlsx", sheet_name=sheet, dtype=str)
    pdf.columns = [snake(c) for c in pdf.columns]
    pdf = pdf.astype(object).where(pdf.notna(), None)
    return (spark.createDataFrame(pdf)
            .withColumn("_source_file", F.lit(f"{stem}.xlsx"))
            .withColumn("_source_sheet", F.lit(sheet))
            .withColumn("_ingested_at", F.current_timestamp()))

for stem, sheets, table in SOURCES:
    try:
        parts = [load_sheet(stem, s) for s in sheets]
    except FileNotFoundError:
        print(f"skipped {stem}: file not uploaded yet")
        continue
    df = reduce(lambda a, b: a.unionByName(b, allowMissingColumns=True), parts)
    (df.write.format("delta").mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(f"clinical_portfolio.bronze.{table}"))
    print(table, df.count())